In [3]:
import os
import re
from pathlib import Path
from typing import List, Tuple, Dict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image


# =========================
# Paths
# =========================
SUPPORT_DIR = r"D:/MSCS_Research/FewShot_Evaluations/Inputs/3Way_5Shot"
QUERY_DIR   = r"D:/MSCS_Research/FewShot_Evaluations/Inputs/Query3"

# =========================
# Device
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================
# Transforms (safe default for both CNNs + ViTs)
# =========================
# Most torchvision ImageNet weights use 224, ImageNet mean/std.
IMG_SIZE = 224
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


# =========================
# I/O utilities
# =========================
def list_images(folder: str) -> List[str]:
    p = Path(folder)
    files = [str(x) for x in p.iterdir() if x.is_file() and x.suffix.lower() in IMG_EXTS]
    files.sort()
    return files

def load_image(path: str) -> torch.Tensor:
    img = Image.open(path).convert("RGB")
    return transform(img)

def normalize_name(s: str) -> str:
    s = Path(s).stem
    s = s.lower().strip()
    s = re.sub(r"\d+$", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


# =========================
# Load support/query
# =========================
def get_support_set(support_dir: str) -> Tuple[torch.Tensor, torch.Tensor, List[str]]:
    support_dir = Path(support_dir)
    class_folders = [p for p in support_dir.iterdir() if p.is_dir()]
    class_folders.sort(key=lambda x: x.name.lower())

    class_names = [p.name for p in class_folders]
    class_to_idx = {name: i for i, name in enumerate(class_names)}

    xs, ys = [], []
    for cf in class_folders:
        imgs = list_images(str(cf))
        for img_path in imgs:
            xs.append(load_image(img_path))
            ys.append(class_to_idx[cf.name])

    X = torch.stack(xs)
    y = torch.tensor(ys, dtype=torch.long)
    return X, y, class_names

def build_query_labels(query_dir: str, class_names: List[str]) -> Tuple[torch.Tensor, torch.Tensor, List[str]]:
    query_files = list_images(query_dir)
    norm_class_to_idx = {normalize_name(cn): i for i, cn in enumerate(class_names)}

    xs, ys, kept = [], [], []
    skipped = []
    for fp in query_files:
        lab = normalize_name(fp)
        if lab in norm_class_to_idx:
            xs.append(load_image(fp))
            ys.append(norm_class_to_idx[lab])
            kept.append(fp)
        else:
            skipped.append(fp)

    if skipped:
        print("\n[WARN] Skipped query images (filename label didn't match a support class):")
        for s in skipped[:20]:
            print("  -", Path(s).name)
        if len(skipped) > 20:
            print(f"  ... (+{len(skipped)-20} more)")

    Xq = torch.stack(xs)
    yq = torch.tensor(ys, dtype=torch.long)
    return Xq, yq, kept


# =========================
# Feature extractors for different architectures
# =========================
class FeatureExtractor(nn.Module):
    """
    Wraps a torchvision model so forward(x) returns a single embedding vector per image: [B, D]
    """
    def __init__(self, backbone: nn.Module, arch: str):
        super().__init__()
        self.backbone = backbone
        self.arch = arch

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        a = self.arch

        # VGG: use features -> pool -> flatten (ignore classifier)
        if a.startswith("vgg"):
            x = self.backbone.features(x)
            x = self.backbone.avgpool(x)
            x = torch.flatten(x, 1)
            return x

        # DenseNet: use features -> relu -> global avg pool -> flatten
        if a.startswith("densenet"):
            x = self.backbone.features(x)
            x = F.relu(x, inplace=True)
            x = F.adaptive_avg_pool2d(x, (1, 1))
            x = torch.flatten(x, 1)
            return x

        # ResNet: use everything except final fc
        if a.startswith("resnet"):
            x = self.backbone.conv1(x)
            x = self.backbone.bn1(x)
            x = self.backbone.relu(x)
            x = self.backbone.maxpool(x)
            x = self.backbone.layer1(x)
            x = self.backbone.layer2(x)
            x = self.backbone.layer3(x)
            x = self.backbone.layer4(x)
            x = self.backbone.avgpool(x)
            x = torch.flatten(x, 1)
            return x

        # EfficientNet / MobileNetV2: use features -> global avg pool
        if a.startswith("efficientnet") or a.startswith("mobilenet_v2"):
            x = self.backbone.features(x)
            x = F.adaptive_avg_pool2d(x, (1, 1))
            x = torch.flatten(x, 1)
            return x

        # ViT (torchvision): forward_features is not public in all versions.
        # Easiest: replace heads with Identity and call the model; output becomes embedding.
        if a.startswith("vit"):
            x = self.backbone(x)  # after head=Identity, this is embedding [B, D]
            return x

        # fallback: try calling the model directly (if head replaced)
        return self.backbone(x)


def build_pretrained_model(name: str) -> Tuple[nn.Module, str]:
    """
    Returns (model, arch_key) where arch_key tells FeatureExtractor how to get embeddings.
    """
    name = name.lower().strip()

    if name == "vgg16":
        m = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        return m, "vgg16"

    if name == "vgg16_bn":
        m = models.vgg16_bn(weights=models.VGG16_BN_Weights.IMAGENET1K_V1)
        return m, "vgg16_bn"

    if name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        return m, "resnet50"

    if name == "resnet101":
        m = models.resnet101(weights=models.ResNet101_Weights.IMAGENET1K_V2)
        return m, "resnet101"

    if name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        return m, "efficientnet_b0"

    if name == "efficientnet_b1":
        m = models.efficientnet_b1(weights=models.EfficientNet_B1_Weights.IMAGENET1K_V1)
        return m, "efficientnet_b1"

    if name == "densenet121":
        m = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        return m, "densenet121"

    if name == "densenet161":
        m = models.densenet161(weights=models.DenseNet161_Weights.IMAGENET1K_V1)
        return m, "densenet161"

    if name == "mobilenet_v2":
        m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V2)
        return m, "mobilenet_v2"

    if name == "vit_b_16":
        m = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
        # remove classifier head so forward returns embeddings
        m.heads = nn.Identity()
        return m, "vit_b_16"

    if name == "vit_b_32":
        m = models.vit_b_32(weights=models.ViT_B_32_Weights.IMAGENET1K_V1)
        m.heads = nn.Identity()
        return m, "vit_b_32"

    raise ValueError(f"Unknown model name: {name}")


# =========================
# Few-shot: prototypes + cosine logits
# =========================
@torch.no_grad()
def compute_prototypes(fe: nn.Module, Xs: torch.Tensor, ys: torch.Tensor, n_way: int) -> torch.Tensor:
    z = fe(Xs)  # [Ns, D]
    protos = torch.stack([z[ys == c].mean(0) for c in range(n_way)], dim=0)  # [Nway, D]
    return protos

@torch.no_grad()
def classify_queries(fe: nn.Module, protos: torch.Tensor, Xq: torch.Tensor, metric: str = "cosine", scale: float = 1.0) -> torch.Tensor:
    zq = fe(Xq)  # [Nq, D]

    if metric == "cosine":
        zq = F.normalize(zq, dim=1)
        protos = F.normalize(protos, dim=1)
        logits = scale * (zq @ protos.t())
    else:
        # negative squared euclidean distance
        q2 = (zq**2).sum(1, keepdim=True)
        p2 = (protos**2).sum(1).unsqueeze(0)
        logits = -(q2 + p2 - 2 * (zq @ protos.t()))

    return logits.argmax(dim=1)

def accuracy(preds: torch.Tensor, y: torch.Tensor) -> float:
    return (preds == y).float().mean().item()


# =========================
# MAIN: evaluate a list of pretrained backbones
# =========================
def main():
    print("Device:", device)

    # Load support/query once
    Xs, ys, class_names = get_support_set(SUPPORT_DIR)
    Xq, yq, qfiles = build_query_labels(QUERY_DIR, class_names)

    Xs, ys = Xs.to(device), ys.to(device)
    Xq, yq = Xq.to(device), yq.to(device)

    print(f"Support: {Xs.shape[0]} images | N-way={len(class_names)}")
    print(f"Query:   {Xq.shape[0]} images")

    model_names = [
        "vgg16", "vgg16_bn",
        "resnet50", "resnet101",
        "efficientnet_b0", "efficientnet_b1",
        "densenet121", "densenet161",
        "mobilenet_v2",
        "vit_b_16", "vit_b_32"
    ]

    results = []

    for name in model_names:
        print(f"\n--- Evaluating {name} ---")
        backbone, arch_key = build_pretrained_model(name)

        backbone.to(device)
        backbone.eval()
        for p in backbone.parameters():
            p.requires_grad = False

        fe = FeatureExtractor(backbone, arch_key).to(device).eval()

        # prototypes + cosine logits (scale can be >1 if you want)
        protos = compute_prototypes(fe, Xs, ys, n_way=len(class_names))
        preds = classify_queries(fe, protos, Xq, metric="cosine", scale=1.0)
        acc = accuracy(preds, yq)

        print(f"Few-shot accuracy (cosine prototypes): {acc*100:.2f}%")
        results.append((name, acc))

    # Sort + print summary
    results.sort(key=lambda x: x[1], reverse=True)
    print("\n========== Summary (best → worst) ==========")
    for name, acc in results:
        print(f"{name:15s}  {acc*100:6.2f}%")

if __name__ == "__main__":
    main()


Device: cpu
Support: 15 images | N-way=3
Query:   40 images

--- Evaluating vgg16 ---
Few-shot accuracy (cosine prototypes): 67.50%

--- Evaluating vgg16_bn ---
Few-shot accuracy (cosine prototypes): 60.00%

--- Evaluating resnet50 ---
Few-shot accuracy (cosine prototypes): 57.50%

--- Evaluating resnet101 ---
Few-shot accuracy (cosine prototypes): 62.50%

--- Evaluating efficientnet_b0 ---
Few-shot accuracy (cosine prototypes): 65.00%

--- Evaluating efficientnet_b1 ---
Few-shot accuracy (cosine prototypes): 77.50%

--- Evaluating densenet121 ---
Few-shot accuracy (cosine prototypes): 65.00%

--- Evaluating densenet161 ---
Few-shot accuracy (cosine prototypes): 65.00%

--- Evaluating mobilenet_v2 ---
Few-shot accuracy (cosine prototypes): 65.00%

--- Evaluating vit_b_16 ---
Few-shot accuracy (cosine prototypes): 62.50%

--- Evaluating vit_b_32 ---
Few-shot accuracy (cosine prototypes): 60.00%

========== Summary (best → worst) ==========
efficientnet_b1   77.50%
vgg16             67.5